In [2]:
import pandas as pd
import json

with open('transactions.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")

         date                     description    category  amount    type
0  2026-03-01  Client Payment - Project Alpha     Revenue   45000  Credit
1  2026-03-02    Vendor: Intermountain Lumber   Inventory  -12000   Debit
2  2026-03-04                 Payroll Service  Operations   -8500   Debit
3  2026-03-05   Client Payment - Project Beta     Revenue   22000  Credit
4  2026-03-08       Lease Payment - Excavator   Equipment   -2200   Debit

Shape: (20, 5)

Data types:
date             str
description      str
category         str
amount         int64
type             str
dtype: object


In [3]:
pivot = df.pivot_table(
    values='amount',
    index='category',
    columns='type',
    aggfunc='sum',
    fill_value=0
)

print("Pivot Table - Amount by Category and Type:")
print(pivot)
print(f"\nTotal by Type:")
print(pivot.sum())

Pivot Table - Amount by Category and Type:
type         Credit  Debit
category                  
Equipment         0  -2200
Growth            0 -10000
Interest        120      0
Inventory         0 -30800
Operations        0 -18750
Revenue      112000      0
Savings           0  -5000
Tech/Growth       0   -650

Total by Type:
type
Credit    112120
Debit     -67400
dtype: int64


# Exploratory Analysis

In [4]:
print("\n1. DATASET OVERVIEW")
print("-" * 60)
print(f"Total Transactions: {len(df)}")
print(f"Date Range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {list(df.columns)}")
print(f"\nDataset Info:")
print(df.info())


1. DATASET OVERVIEW
------------------------------------------------------------
Total Transactions: 20
Date Range: 2026-03-01 to 2026-03-31
Columns: ['date', 'description', 'category', 'amount', 'type']

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   date         20 non-null     str  
 1   description  20 non-null     str  
 2   category     20 non-null     str  
 3   amount       20 non-null     int64
 4   type         20 non-null     str  
dtypes: int64(1), str(4)
memory usage: 932.0 bytes
None


In [5]:
print("\n2. MISSING VALUES CHECK")
print("-" * 60)
print(df.isnull().sum())


2. MISSING VALUES CHECK
------------------------------------------------------------
date           0
description    0
category       0
amount         0
type           0
dtype: int64


In [6]:
print("\n3. STATISTICAL SUMMARY")
print("-" * 60)
print(df['amount'].describe())


3. STATISTICAL SUMMARY
------------------------------------------------------------
count       20.000000
mean      2236.000000
std      14698.523879
min     -12000.000000
25%      -5875.000000
50%       -550.000000
75%        -82.500000
max      45000.000000
Name: amount, dtype: float64


In [7]:
print("\n4. TRANSACTION TYPE BREAKDOWN")
print("-" * 60)
type_summary = df.groupby('type').agg({
    'amount': ['count', 'sum', 'mean', 'min', 'max']
}).round(2)
print(type_summary)


4. TRANSACTION TYPE BREAKDOWN
------------------------------------------------------------
       amount                                
        count     sum      mean    min    max
type                                         
Credit      5  112120  22424.00    120  45000
Debit      15  -67400  -4493.33 -12000   -150


In [8]:
print("\n5. CATEGORY BREAKDOWN")
print("-" * 60)
category_summary = df.groupby('category').agg({
    'amount': ['count', 'sum', 'mean', 'min', 
               lambda x: x.quantile(0.25),  
               lambda x: x.quantile(0.50),  
               lambda x: x.quantile(0.75),  
               'max']
}).round(2)
category_summary.columns = ['count', 'sum', 'mean', 'min', 'p25', 'p50', 'p75', 'max']
category_summary = category_summary.sort_values('sum', ascending=False)
print(category_summary)


5. CATEGORY BREAKDOWN
------------------------------------------------------------
             count     sum     mean    min      p25      p50      p75    max
category                                                                    
Revenue          4  112000  28000.0  15000  20250.0  26000.0  33750.0  45000
Interest         1     120    120.0    120    120.0    120.0    120.0    120
Tech/Growth      2    -650   -325.0   -500   -412.5   -325.0   -237.5   -150
Equipment        1   -2200  -2200.0  -2200  -2200.0  -2200.0  -2200.0  -2200
Savings          1   -5000  -5000.0  -5000  -5000.0  -5000.0  -5000.0  -5000
Growth           1  -10000 -10000.0 -10000 -10000.0 -10000.0 -10000.0 -10000
Operations       6  -18750  -3125.0  -8500  -6525.0   -525.0   -412.5   -300
Inventory        4  -30800  -7700.0 -12000 -11250.0  -7500.0  -3950.0  -3800


In [9]:
print("\n6. TOP 5 LARGEST CREDITS")
print("-" * 60)
print(df[df['type'] == 'Credit'].nlargest(7, 'amount')[['date', 'description', 'category', 'amount']])

print("\n7. TOP 5 LARGEST DEBITS (by absolute value)")
print("-" * 60)
print(df[df['type'] == 'Debit'].nsmallest(5, 'amount')[['date', 'description', 'category', 'amount']])



6. TOP 5 LARGEST CREDITS
------------------------------------------------------------
          date                     description  category  amount
0   2026-03-01  Client Payment - Project Alpha   Revenue   45000
14  2026-03-27  Client Payment - Project Gamma   Revenue   30000
3   2026-03-05   Client Payment - Project Beta   Revenue   22000
7   2026-03-15        Client Payment - Deposit   Revenue   15000
10  2026-03-22           Bonus Interest Earned  Interest     120

7. TOP 5 LARGEST DEBITS (by absolute value)
------------------------------------------------------------
          date                   description    category  amount
1   2026-03-02  Vendor: Intermountain Lumber   Inventory  -12000
11  2026-03-23  Vendor: Intermountain Lumber   Inventory  -11000
19  2026-03-31     New Equipment Downpayment      Growth  -10000
2   2026-03-04               Payroll Service  Operations   -8500
12  2026-03-25               Payroll Service  Operations   -8500


In [10]:
print("\n8. FINANCIAL SUMMARY")
print("-" * 60)
total_credits = df[df['type'] == 'Credit']['amount'].sum()
total_debits = df[df['type'] == 'Debit']['amount'].sum()
net_amount = total_credits + total_debits

print(f"Total Credits:  ${total_credits:.2f}")
print(f"Total Debits:   ${total_debits:.2f}")
print(f"Net Amount:     ${net_amount:.2f}")


8. FINANCIAL SUMMARY
------------------------------------------------------------
Total Credits:  $112120.00
Total Debits:   $-67400.00
Net Amount:     $44720.00


In [11]:
print("\n9. TRANSACTION COUNT BY CATEGORY")
print("-" * 60)
print(df['category'].value_counts().sort_values(ascending=False))


9. TRANSACTION COUNT BY CATEGORY
------------------------------------------------------------
category
Operations     6
Revenue        4
Inventory      4
Tech/Growth    2
Equipment      1
Interest       1
Savings        1
Growth         1
Name: count, dtype: int64


In [ ]:
retag_view = (
    df[['category', 'description']]
    .dropna(subset=['category', 'description'])
    .assign(description=lambda x: x['description'].astype(str).str.strip())
    .query("description != ''")
    .groupby('category')['description']
    .apply(lambda s: sorted(s.unique()))
    .reset_index(name='unique_descriptions')
)

retag_view['unique_count'] = retag_view['unique_descriptions'].str.len()
retag_view = retag_view.sort_values(['unique_count', 'category'], ascending=[False, True])

print("Unique description values by category:")
for _, row in retag_view.iterrows():
    print(f"\n{row['category']} ({row['unique_count']} unique descriptions)")
    for d in row['unique_descriptions']:
        print(f"  - {d}")

retag_view

Unique description values by category:

Operations (5 unique descriptions)
  - Gas & Fuel
  - Payroll Service
  - Site Permit Fees
  - Tool Repair
  - Utility Bill

Revenue (4 unique descriptions)
  - Client Payment - Deposit
  - Client Payment - Project Alpha
  - Client Payment - Project Beta
  - Client Payment - Project Gamma

Inventory (2 unique descriptions)
  - Vendor: Intermountain Lumber
  - Vendor: Steel Supply Co

Tech/Growth (2 unique descriptions)
  - Marketing - Local SEO
  - Software Subscription (AutoCAD)

Equipment (1 unique descriptions)
  - Lease Payment - Excavator

Growth (1 unique descriptions)
  - New Equipment Downpayment

Interest (1 unique descriptions)
  - Bonus Interest Earned

Savings (1 unique descriptions)
  - Internal Transfer to Savings


,category,unique_descriptions,unique_count
4,Operations,"[Gas & Fuel, Payroll Service, Site Permit Fees...",5
5,Revenue,"[Client Payment - Deposit, Client Payment - Pr...",4
3,Inventory,"[Vendor: Intermountain Lumber, Vendor: Steel S...",2
7,Tech/Growth,"[Marketing - Local SEO, Software Subscription ...",2
0,Equipment,[Lease Payment - Excavator],1
1,Growth,[New Equipment Downpayment],1
2,Interest,[Bonus Interest Earned],1
6,Savings,[Internal Transfer to Savings],1
